# 01 — Chicago Crime Data Exploration

**Capstone:** Chicago Police Department Crime Analytics  
**Database choice:** SQLite3 (not MySQL)

This notebook begins Use Case 1 from the assignment: load the master crime CSV, inspect the first 10 records, identify the number of rows and columns, inspect data types, and review missing values.

The crime source file does not contain `primary_type` directly. That description is stored in `iucr_codes.csv` and will be joined in a later step.

## 1. Import libraries and set project paths

Run this notebook from the project root or from the `notebooks` folder. The path logic below supports both locations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the project root whether this notebook is opened from the root or notebooks folder.
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data" / "raw").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "raw").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError("Could not find data/raw. Open the chicago_crime_capstone project folder in VS Code.")

DATA_DIR = PROJECT_DIR / "data" / "raw"
CRIME_FILE = DATA_DIR / "chicago_crime_dataset.csv"
IUCR_FILE = DATA_DIR / "iucr_codes.csv"

print("Project folder:", PROJECT_DIR)
print("Crime file exists:", CRIME_FILE.exists())
print("IUCR file exists:", IUCR_FILE.exists())

Project folder: c:\Users\deepe\Downloads\chicago_crime_capstone
Crime file exists: True
IUCR file exists: True


## 2. Load the crime dataset

The supplied crime CSV has an issue in its final two records: the quoted `location` field is missing its closing quotation mark. The code below repairs only lines with an odd number of quote characters, then reads the file with Pandas. We keep the original file unchanged.

In [8]:
import io

def read_chicago_crime_csv(path):

    text = path.read_text(
        encoding="utf-8-sig",
        errors="replace"
    )

    fixed_lines = []
    repaired_line_numbers = []

    for line_number, line in enumerate(
        text.splitlines(), start=1
    ):

        if line.count('"') % 2 == 1:
            line = line + '"'
            repaired_line_numbers.append(line_number)

        fixed_lines.append(line)

    csv_text = "\n".join(fixed_lines)

    frame = pd.read_csv(
        io.StringIO(csv_text),
        low_memory=False
    )

    if frame.shape[1] != 20:
        raise ValueError(
            f"Expected 20 columns, got {frame.shape[1]}"
        )

    return frame, repaired_line_numbers


crime_df, repaired_lines = read_chicago_crime_csv(
    CRIME_FILE
)

print("Dataset loaded successfully.")
print("Rows:", crime_df.shape[0])
print("Columns:", crime_df.shape[1])
print("Repaired lines:", repaired_lines)

Dataset loaded successfully.
Rows: 498
Columns: 20
Repaired lines: [498, 499]


## 3. Display the first 10 rows

In [9]:
crime_df.head(10)

,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"


## 4. Inspect column names, data types and dataset dimensions

In [10]:
print("Dataset shape:", crime_df.shape)
print("\\nColumn names:")
print(crime_df.columns.tolist())

print("\\nData types:")
display(crime_df.dtypes.to_frame(name="data_type"))

print("\\nDataFrame information:")
crime_df.info()

Dataset shape: (498, 20)
\nColumn names:
['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location']
\nData types:


,data_type
id,int64
case_number,str
date,str
block,str
iucr_code,str
location_desc,str
arrest,bool
domestic,bool
beat_num,int64
district_code,int64


\nDataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              498 non-null    int64  
 1   case_number     498 non-null    str    
 2   date            498 non-null    str    
 3   block           498 non-null    str    
 4   iucr_code       498 non-null    str    
 5   location_desc   494 non-null    str    
 6   arrest          498 non-null    bool   
 7   domestic        498 non-null    bool   
 8   beat_num        498 non-null    int64  
 9   district_code   498 non-null    int64  
 10  ward_no         498 non-null    int64  
 11  community_code  498 non-null    int64  
 12  fbi_code        498 non-null    str    
 13  x_coordinate    469 non-null    float64
 14  y_coordinate    469 non-null    float64
 15  year            498 non-null    int64  
 16  date_of_update  498 non-null    str    
 17  latitude        469 n

## 5. Missing values and duplicate records

In [11]:
missing_summary = pd.DataFrame({
    "missing_count": crime_df.isna().sum(),
    "missing_percent": (crime_df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

print("Exact duplicate rows:", crime_df.duplicated().sum())
if "id" in crime_df.columns:
    print("Duplicate crime IDs:", crime_df["id"].duplicated().sum())

,missing_count,missing_percent
location,29,5.82
y_coordinate,29,5.82
longitude,29,5.82
latitude,29,5.82
x_coordinate,29,5.82
location_desc,4,0.80
case_number,0,0.00
id,0,0.00
block,0,0.00
date,0,0.00


Exact duplicate rows: 0
Duplicate crime IDs: 0


## 6. Load IUCR reference data and identify crime categories

The crime file stores `iucr_code`. The separate IUCR reference file maps each code to its primary crime type and description. We standardize the column names and join these files to answer the assignment question about the number of unique crime types.

In [12]:
iucr_df = pd.read_csv(IUCR_FILE, low_memory=False)

# Standardize reference-file column names (including trailing spaces).
iucr_df.columns = (
    iucr_df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

crime_df["iucr_code"] = crime_df["iucr_code"].astype("string").str.strip()
iucr_df["iucr"] = iucr_df["iucr"].astype("string").str.strip()

crime_with_type = crime_df.merge(
    iucr_df[["iucr", "primary_type", "description"]],
    left_on="iucr_code",
    right_on="iucr",
    how="left"
)

print("Unique IUCR codes in crime data:", crime_df["iucr_code"].nunique())
print("Unique primary crime types matched from IUCR:", crime_with_type["primary_type"].nunique())
print("Rows without a matching IUCR description:", crime_with_type["primary_type"].isna().sum())

display(
    crime_with_type["primary_type"]
    .value_counts(dropna=False)
    .head(15)
    .rename_axis("primary_type")
    .reset_index(name="record_count")
)

Unique IUCR codes in crime data: 97
Unique primary crime types matched from IUCR: 23
Rows without a matching IUCR description: 0


,primary_type,record_count
0,THEFT,105
1,BATTERY,85
2,CRIMINAL DAMAGE,49
3,OTHER OFFENSE,39
4,DECEPTIVE PRACTICE,38
5,ASSAULT,35
6,NARCOTICS,32
7,ROBBERY,30
8,CRIMINAL TRESPASS,17
9,MOTOR VEHICLE THEFT,16


In [13]:
# Convert date columns to datetime
crime_df["date"] = pd.to_datetime(
    crime_df["date"],
    errors="coerce"
)

crime_df["date_of_update"] = pd.to_datetime(
    crime_df["date_of_update"],
    errors="coerce"
)

# Create derived date columns
crime_df["Year"] = crime_df["date"].dt.year
crime_df["Month"] = crime_df["date"].dt.month
crime_df["DayOfWeek"] = crime_df["date"].dt.day_name()

# Display results
display(
    crime_df[
        ["date", "Year", "Month", "DayOfWeek"]
    ].head(10)
)

print("Date conversion completed.")
print("\nDate column types:")
print(crime_df[["date", "date_of_update"]].dtypes)

,date,Year,Month,DayOfWeek
0,2016-05-03,2016,5,Tuesday
1,2016-05-03,2016,5,Tuesday
2,2016-05-03,2016,5,Tuesday
3,2016-05-03,2016,5,Tuesday
4,2016-05-03,2016,5,Tuesday
5,2016-05-03,2016,5,Tuesday
6,2016-05-03,2016,5,Tuesday
7,2016-05-03,2016,5,Tuesday
8,2016-05-03,2016,5,Tuesday
9,2016-05-03,2016,5,Tuesday


Date conversion completed.

Date column types:
date              datetime64[us]
date_of_update    datetime64[us]
dtype: object


In [14]:
# Check whether date conversion created missing values

date_missing = crime_df[
    ["date", "date_of_update", "Year", "Month", "DayOfWeek"]
].isnull().sum()

print("Missing values after date conversion:")
display(date_missing)

Missing values after date conversion:


date              0
date_of_update    0
Year              0
Month             0
DayOfWeek         0
dtype: int64

In [16]:
print(crime_df.columns.tolist())

['id', 'case_number', 'date', 'block', 'iucr_code', 'location_desc', 'arrest', 'domestic', 'beat_num', 'district_code', 'ward_no', 'community_code', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'date_of_update', 'latitude', 'longitude', 'location', 'Year', 'Month', 'DayOfWeek']


In [17]:
print(iucr_df.columns.tolist())
display(iucr_df.head())

['iucr', 'primary_type', 'description', 'index_code']


,iucr,primary_type,description,index_code
0,110,HOMICIDE,FIRST DEGREE MURDER,I
1,130,HOMICIDE,SECOND DEGREE MURDER,I
2,141,HOMICIDE,INVOLUNTARY MANSLAUGHTER,N
3,142,HOMICIDE,RECKLESS HOMICIDE,N
4,261,CRIMINAL SEXUAL ASSAULT,AGGRAVATED - HANDGUN,I


In [18]:
# Prepare the IUCR lookup table
iucr_lookup = iucr_df[
    ["iucr", "primary_type"]
].drop_duplicates(subset=["iucr"])

# Standardize the join keys
crime_df["iucr_code"] = (
    crime_df["iucr_code"]
    .astype("string")
    .str.strip()
)

iucr_lookup["iucr"] = (
    iucr_lookup["iucr"]
    .astype("string")
    .str.strip()
)

# Merge crime categories into the main dataset
crime_df = crime_df.merge(
    iucr_lookup,
    left_on="iucr_code",
    right_on="iucr",
    how="left"
)

# Remove the extra IUCR column
crime_df.drop(columns=["iucr"], inplace=True)

print("IUCR mapping completed.")

display(
    crime_df[["iucr_code", "primary_type"]].head(10)
)

IUCR mapping completed.


C:\Users\deepe\AppData\Local\Temp\ipykernel_14496\1785809308.py:28: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  crime_df.drop(columns=["iucr"], inplace=True)


,iucr_code,primary_type
0,486,BATTERY
1,486,BATTERY
2,470,PUBLIC PEACE VIOLATION
3,460,BATTERY
4,820,THEFT
5,041A,BATTERY
6,460,BATTERY
7,460,BATTERY
8,460,BATTERY
9,486,BATTERY


In [19]:
# Identify unmatched IUCR codes
unmatched_codes = crime_df[
    crime_df["primary_type"].isna()
]["iucr_code"].unique()

print("Unmatched IUCR codes:", len(unmatched_codes))
print(unmatched_codes)

print("\nPrimary crime category counts:")
display(
    crime_df["primary_type"].value_counts(dropna=False)
)

Unmatched IUCR codes: 0
<ArrowStringArray>
[]
Length: 0, dtype: string

Primary crime category counts:


primary_type
THEFT                               105
BATTERY                              85
CRIMINAL DAMAGE                      49
OTHER OFFENSE                        39
DECEPTIVE PRACTICE                   38
ASSAULT                              35
NARCOTICS                            32
ROBBERY                              30
CRIMINAL TRESPASS                    17
MOTOR VEHICLE THEFT                  16
BURGLARY                             14
WEAPONS VIOLATION                     8
PUBLIC PEACE VIOLATION                5
OFFENSE INVOLVING CHILDREN            5
CRIMINAL SEXUAL ASSAULT               4
SEX OFFENSE                           3
HOMICIDE                              3
PROSTITUTION                          3
KIDNAPPING                            2
INTERFERENCE WITH PUBLIC OFFICER      2
STALKING                              1
OBSCENITY                             1
GAMBLING                              1
Name: count, dtype: int64

In [20]:
categorical_columns = [
    "case_number",
    "block",
    "iucr_code",
    "location_desc",
    "fbi_code",
    "DayOfWeek",
    "primary_type"
]

for column in categorical_columns:
    crime_df[column] = (
        crime_df[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

print("Categorical values standardized.")

print("\nPrimary crime categories:")
display(crime_df["primary_type"].value_counts())

Categorical values standardized.

Primary crime categories:


primary_type
THEFT                               105
BATTERY                              85
CRIMINAL DAMAGE                      49
OTHER OFFENSE                        39
DECEPTIVE PRACTICE                   38
ASSAULT                              35
NARCOTICS                            32
ROBBERY                              30
CRIMINAL TRESPASS                    17
MOTOR VEHICLE THEFT                  16
BURGLARY                             14
WEAPONS VIOLATION                     8
PUBLIC PEACE VIOLATION                5
OFFENSE INVOLVING CHILDREN            5
CRIMINAL SEXUAL ASSAULT               4
SEX OFFENSE                           3
HOMICIDE                              3
PROSTITUTION                          3
KIDNAPPING                            2
INTERFERENCE WITH PUBLIC OFFICER      2
STALKING                              1
OBSCENITY                             1
GAMBLING                              1
Name: count, dtype: int64[pyarrow]

In [22]:
# Calculate missing values
missing_summary = pd.DataFrame({
    "Missing Count": crime_df.isnull().sum(),
    "Missing Percentage": (
        crime_df.isnull().sum() / len(crime_df) * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

display(missing_summary)

,Missing Count,Missing Percentage
latitude,29,5.82
longitude,29,5.82
x_coordinate,29,5.82
location,29,5.82
y_coordinate,29,5.82
location_desc,4,0.80
date,0,0.00
id,0,0.00
domestic,0,0.00
arrest,0,0.00


In [23]:
print("Dataset shape:", crime_df.shape)

print(
    "Duplicate rows:",
    crime_df.duplicated().sum()
)

print(
    "Duplicate crime IDs:",
    crime_df["id"].duplicated().sum()
)

print(
    "Missing primary_type values:",
    crime_df["primary_type"].isna().sum()
)

Dataset shape: (498, 24)
Duplicate rows: 0
Duplicate crime IDs: 0
Missing primary_type values: 0


In [24]:
from pathlib import Path

# Create processed data directory
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned dataset
output_file = PROCESSED_DIR / "chicago_crime_cleaned.csv"

crime_df.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully.")
print("File location:", output_file.resolve())

Cleaned dataset saved successfully.
File location: C:\Users\deepe\Downloads\chicago_crime_capstone\data\processed\chicago_crime_cleaned.csv
